# Assignment 6 — Smoothing Techniques for N-gram Language Models

This notebook implements 5 smoothing techniques on top of the n-gram counts trained in Assignment 4,
and evaluates every model on the **development** and **test** sets (1,000 Hindi sentences each).

1. **Interpolated Smoothing** (Bigram, Trigram, Quadrigram) — Jelinek-Mercer interpolation, weights tuned on the dev set
2. **Good Turing Smoothing** — count-of-counts rescaling with cutoff k = 5
3. **Katz Backoff Smoothing** — Good-Turing discounted counts backed off with alpha weights
4. **Stupid Backoff** — unnormalized backoff scores with discount factor 0.4 (Brants & Franz, 2009)
5. **Kneser-Ney Smoothing** — absolute discounting with continuation probabilities

All models report log probability, tokens, cross entropy (bits), and perplexity. For Stupid Backoff the scores are
unnormalized, so its "perplexity" (2 ^ average −log2 score) is a comparable ranking metric rather than a true
probability-domain perplexity.

**Unseen events:** the dev/test sets contain out-of-vocabulary words. Every technique therefore assigns a small
non-zero base probability to unseen unigrams, `UNSEEN_UNIGRAM_PROBABILITY = N1 / (N · V)` (the Good-Turing
unseen mass spread over V unseen types), so all reported values stay finite.

In [1]:
import os
import pickle
import math
import pandas as pd
from collections import Counter

In [2]:
MODELS_PATH = "../Assignment_4/models"
DATA_PATH = "../Assignment_4/data"

os.makedirs("results", exist_ok=True)

In [3]:
with open(os.path.join(MODELS_PATH, "unigram_counts.pkl"), "rb") as f:
    unigram_counts = pickle.load(f)

with open(os.path.join(MODELS_PATH, "bigram_counts.pkl"), "rb") as f:
    bigram_counts = pickle.load(f)

with open(os.path.join(MODELS_PATH, "trigram_counts.pkl"), "rb") as f:
    trigram_counts = pickle.load(f)

with open(os.path.join(MODELS_PATH, "quadrigram_counts.pkl"), "rb") as f:
    quadrigram_counts = pickle.load(f)

with open(os.path.join(MODELS_PATH, "vocabulary.pkl"), "rb") as f:
    vocabulary = pickle.load(f)

# Context (prefix) occurrence counts saved by Assignment 4.
with open(os.path.join(MODELS_PATH, "bigram_context_counts.pkl"), "rb") as f:
    bigram_context_counts = pickle.load(f)

with open(os.path.join(MODELS_PATH, "trigram_context_counts.pkl"), "rb") as f:
    trigram_context_counts = pickle.load(f)

with open(os.path.join(MODELS_PATH, "quadrigram_context_counts.pkl"), "rb") as f:
    quadrigram_context_counts = pickle.load(f)

V = len(vocabulary)
total_unigram_tokens = sum(unigram_counts.values())

print(f"Vocabulary size (V): {V:,}")
print(f"Total unigram tokens: {total_unigram_tokens:,}")
print(f"Unique unigrams:     {len(unigram_counts):,}")
print(f"Unique bigrams:      {len(bigram_counts):,}")
print(f"Unique trigrams:     {len(trigram_counts):,}")
print(f"Unique quadrigrams:  {len(quadrigram_counts):,}")

Vocabulary size (V): 307,861
Total unigram tokens: 24,079,766
Unique unigrams:     307,861
Unique bigrams:      3,782,573
Unique trigrams:     10,720,662
Unique quadrigrams:  15,632,208


In [4]:
train_df = pd.read_parquet(os.path.join(DATA_PATH, "train.parquet"))
dev_df = pd.read_parquet(os.path.join(DATA_PATH, "dev.parquet"))
test_df = pd.read_parquet(os.path.join(DATA_PATH, "test.parquet"))

def prepare_sentence(tokens):
    return ["<s>"] + list(tokens) + ["</s>"]

train_sentences = [prepare_sentence(t) for t in train_df["tokens"]]
dev_sentences = [prepare_sentence(t) for t in dev_df["tokens"]]
test_sentences = [prepare_sentence(t) for t in test_df["tokens"]]

print("Training sentences:", len(train_sentences))
print("Development sentences:", len(dev_sentences))
print("Test sentences:", len(test_sentences))

Training sentences: 998000
Development sentences: 1000
Test sentences: 1000


## Shared Building Blocks

Helpers used by all five techniques: context counts per order, the MLE probability, and a non-zero base
probability for unseen unigrams.

In [5]:
COUNT_MAPS = {1: unigram_counts, 2: bigram_counts, 3: trigram_counts, 4: quadrigram_counts}

# Assignment 4 stores bigram context counts keyed by the word string; rekey to 1-tuples
# so lookups with ngram[:-1] work uniformly at every order.
bigram_context_counts = {(w1,): count for w1, count in bigram_context_counts.items()}

CONTEXT_COUNTS = {
    2: bigram_context_counts,
    3: trigram_context_counts,
    4: quadrigram_context_counts,
}

TOTAL_TOKENS = {
    1: total_unigram_tokens,
    2: sum(bigram_counts.values()),
    3: sum(trigram_counts.values()),
    4: sum(quadrigram_counts.values()),
}

COUNT_OF_COUNTS = {
    1: Counter(unigram_counts.values()),
    2: Counter(bigram_counts.values()),
    3: Counter(trigram_counts.values()),
    4: Counter(quadrigram_counts.values()),
}

# Good-Turing unseen mass: N1 / (N * V) per unseen type, at every order.
UNSEEN_PROBABILITY = {
    n: COUNT_OF_COUNTS[n].get(1, 0) / (TOTAL_TOKENS[n] * V) for n in (1, 2, 3, 4)
}


def unigram_probability(word):
    count = unigram_counts.get((word,), 0)
    if count > 0:
        return count / total_unigram_tokens
    return UNSEEN_PROBABILITY[1]


def mle_probability(n, ngram):
    """Raw MLE probability for n >= 2; zero when the n-gram or its context is unseen."""
    count = COUNT_MAPS[n].get(ngram, 0)
    context_count = CONTEXT_COUNTS[n].get(ngram[:-1], 0)
    if count > 0 and context_count > 0:
        return count / context_count
    return 0.0


print("Unseen-event base probability per order:")
for n in (1, 2, 3, 4):
    print(f"  order {n}: {UNSEEN_PROBABILITY[n]:.3e}")

Unseen-event base probability per order:
  order 1: 2.258e-08
  order 2: 3.654e-07
  order 3: 1.304e-06
  order 4: 2.196e-06


## 1. Interpolated Smoothing (Jelinek-Mercer)

Each order is interpolated with the recursively interpolated lower order:

    P_interp(w|h) = λ_n · P_mle(w|h) + (1 − λ_n) · P_interp(w|h')

with the unigram distribution (including the unseen floor) as the base case. The weights λ2, λ3, λ4 are
**tuned on the development set** by grid search over total log probability, sequentially: λ2 for the bigram
model, then λ3 (with λ2 fixed), then λ4 (with λ3, λ2 fixed).

In [6]:
def interpolated_probability(n, ngram, lambdas):
    """lambdas = (lambda_n, ..., lambda_2); the unigram base case needs no weight."""
    if n == 1:
        return unigram_probability(ngram[0])

    mle = mle_probability(n, ngram)
    lower = interpolated_probability(n - 1, ngram[1:], lambdas[1:])
    return lambdas[0] * mle + (1 - lambdas[0]) * lower


def dev_log_probability_interpolated(n, lambdas):
    total = 0.0
    for sentence in dev_sentences:
        for i in range(n - 1, len(sentence)):
            ngram = tuple(sentence[i - n + 1:i + 1])
            total += math.log2(max(interpolated_probability(n, ngram, lambdas), 1e-300))
    return total

In [7]:
best_l2, best_score = 0.0, -float("inf")
for l2 in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    score = dev_log_probability_interpolated(2, (l2,))
    print(f"lambda2 = {l2:.1f}  ->  dev log probability = {score:,.2f}")
    if score > best_score:
        best_l2, best_score = l2, score

LAMBDA2 = best_l2
print(f"\nSelected lambda2 = {LAMBDA2:.1f}")

lambda2 = 0.0  ->  dev log probability = -255,450.11


lambda2 = 0.1  ->  dev log probability = -224,676.19
lambda2 = 0.2  ->  dev log probability = -214,569.10


lambda2 = 0.3  ->  dev log probability = -208,069.86


lambda2 = 0.4  ->  dev log probability = -203,385.31
lambda2 = 0.5  ->  dev log probability = -199,873.68


lambda2 = 0.6  ->  dev log probability = -197,261.69


lambda2 = 0.7  ->  dev log probability = -195,472.09
lambda2 = 0.8  ->  dev log probability = -194,638.43


lambda2 = 0.9  ->  dev log probability = -195,479.03


lambda2 = 1.0  ->  dev log probability = -3,267,575.54

Selected lambda2 = 0.8


In [8]:
best_l3, best_score = 0.0, -float("inf")
for l3 in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    score = dev_log_probability_interpolated(3, (l3, LAMBDA2))
    print(f"lambda3 = {l3:.1f}  ->  dev log probability = {score:,.2f}")
    if score > best_score:
        best_l3, best_score = l3, score

LAMBDA3 = best_l3
print(f"\nSelected lambda3 = {LAMBDA3:.1f} (lambda2 = {LAMBDA2:.1f})")

lambda3 = 0.0  ->  dev log probability = -183,328.74


lambda3 = 0.1  ->  dev log probability = -177,009.66


lambda3 = 0.2  ->  dev log probability = -175,517.40


lambda3 = 0.3  ->  dev log probability = -175,142.82


lambda3 = 0.4  ->  dev log probability = -175,526.90


...

lambda3 = 0.6  ->  dev log probability = -178,491.05


lambda3 = 0.7  ->  dev log probability = -181,458.36


lambda3 = 0.8  ->  dev log probability = -186,246.06


lambda3 = 0.9  ->  dev log probability = -195,273.17


lambda3 = 1.0  ->  dev log probability = -9,952,600.14

Selected lambda3 = 0.3 (lambda2 = 0.8)


In [9]:
best_l4, best_score = 0.0, -float("inf")
for l4 in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    score = dev_log_probability_interpolated(4, (l4, LAMBDA3, LAMBDA2))
    print(f"lambda4 = {l4:.1f}  ->  dev log probability = {score:,.2f}")
    if score > best_score:
        best_l4, best_score = l4, score

LAMBDA4 = best_l4
print(f"\nSelected lambda4 = {LAMBDA4:.1f} (lambda3 = {LAMBDA3:.1f}, lambda2 = {LAMBDA2:.1f})")

lambda4 = 0.0  ->  dev log probability = -167,660.78
lambda4 = 0.1  ->  dev log probability = -167,900.87


lambda4 = 0.2  ->  dev log probability = -169,289.62
lambda4 = 0.3  ->  dev log probability = -171,304.36


lambda4 = 0.4  ->  dev log probability = -173,939.14
lambda4 = 0.5  ->  dev log probability = -177,311.49


lambda4 = 0.6  ->  dev log probability = -181,675.39
lambda4 = 0.7  ->  dev log probability = -187,542.32


lambda4 = 0.8  ->  dev log probability = -196,088.31
lambda4 = 0.9  ->  dev log probability = -211,095.06


lambda4 = 1.0  ->  dev log probability = -15,420,366.64

Selected lambda4 = 0.0 (lambda3 = 0.3, lambda2 = 0.8)


In [10]:
def evaluate_interpolated(sentences, n, lambdas):
    total = 0.0
    tokens = 0
    for sentence in sentences:
        for i in range(n - 1, len(sentence)):
            ngram = tuple(sentence[i - n + 1:i + 1])
            total += math.log2(max(interpolated_probability(n, ngram, lambdas), 1e-300))
            tokens += 1
    ce = -total / tokens
    return total, tokens, ce, 2 ** ce


INTERPOLATED_MODELS = [
    ("Bigram", 2, (LAMBDA2,)),
    ("Trigram", 3, (LAMBDA3, LAMBDA2)),
    ("Quadrigram", 4, (LAMBDA4, LAMBDA3, LAMBDA2)),
]


def results_table(sentences, models):
    rows = []
    for name, n, lambdas in models:
        log_p, tokens, ce, perplexity = evaluate_interpolated(sentences, n, lambdas)
        rows.append({
            "Model": name,
            "Log Probability": log_p,
            "Tokens": tokens,
            "Cross Entropy": ce,
            "Perplexity": perplexity,
        })
    return pd.DataFrame(rows)

In [11]:
interp_dev_results = results_table(dev_sentences, INTERPOLATED_MODELS)
interp_dev_results.to_csv("results/interpolated_development_results.csv", index=False)
interp_dev_results

,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Bigram,-194638.425378,23954,8.125508,279.268352
1,Trigram,-175142.822208,22954,7.630166,198.111064
2,Quadrigram,-167660.783670,21954,7.636913,199.039755


In [12]:
interp_test_results = results_table(test_sentences, INTERPOLATED_MODELS)
interp_test_results.to_csv("results/interpolated_test_results.csv", index=False)
interp_test_results

,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Bigram,-213882.097629,26068,8.204776,295.041867
1,Trigram,-193158.577227,25068,7.705384,208.714127
2,Quadrigram,-185359.706540,24068,7.701500,208.152949


## 2. Good Turing Smoothing

Good Turing re-estimates counts from the count of counts `N_c` (number of n-gram types seen exactly c times):

    c* = (c + 1) · N_{c+1} / N_c

- Counts above the cutoff (k = 5) are left unchanged.
- **Cap:** for small c the raw GT estimate can exceed c (when N_{c+1} is large), which would produce
  probabilities above 1; c* is therefore capped at c.
- The unseen mass `N1 / (N · V)` is assigned to every unseen type of each order.
- An unseen context backs off to the next lower order GT estimate, so every token receives a non-zero
  probability.

In [13]:
GT_CUTOFF = 5


def gt_adjusted_count(n, count):
    if count >= GT_CUTOFF or count <= 0:
        return count
    n_c = COUNT_OF_COUNTS[n].get(count, 0)
    n_c1 = COUNT_OF_COUNTS[n].get(count + 1, 0)
    if n_c == 0:
        return count
    return min(count, (count + 1) * n_c1 / n_c)


def gt_probability(n, ngram):
    if n == 1:
        return unigram_probability(ngram[0])

    context_count = CONTEXT_COUNTS[n].get(ngram[:-1], 0)
    if context_count == 0:
        return gt_probability(n - 1, ngram[1:])

    count = COUNT_MAPS[n].get(ngram, 0)
    if count == 0:
        return UNSEEN_PROBABILITY[n]

    return max(gt_adjusted_count(n, count), 1e-300) / context_count

In [14]:
def evaluate_gt(sentences, n):
    total = 0.0
    tokens = 0
    for sentence in sentences:
        for i in range(n - 1, len(sentence)):
            ngram = tuple(sentence[i - n + 1:i + 1])
            total += math.log2(max(gt_probability(n, ngram), 1e-300))
            tokens += 1
    ce = -total / tokens
    return total, tokens, ce, 2 ** ce


GT_MODELS = [
    ("Unigram", 1),
    ("Bigram", 2),
    ("Trigram", 3),
    ("Quadrigram", 4),
]


def results_table_gt(sentences):
    rows = []
    for name, n in GT_MODELS:
        log_p, tokens, ce, perplexity = evaluate_gt(sentences, n)
        rows.append({
            "Model": name,
            "Log Probability": log_p,
            "Tokens": tokens,
            "Cross Entropy": ce,
            "Perplexity": perplexity,
        })
    return pd.DataFrame(rows)

In [15]:
gt_dev_results = results_table_gt(dev_sentences)
gt_dev_results.to_csv("results/good_turing_development_results.csv", index=False)
gt_dev_results

,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Unigram,-260042.748092,24954,10.420884,1370.877887
1,Bigram,-200405.346086,23954,8.366258,329.985331
2,Trigram,-217567.384557,22954,9.478408,713.321337
3,Quadrigram,-236550.116336,21954,10.774807,1752.024407


In [16]:
gt_test_results = results_table_gt(test_sentences)
gt_test_results.to_csv("results/good_turing_test_results.csv", index=False)
gt_test_results

,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Unigram,-286410.827649,27068,10.581160,1531.956466
1,Bigram,-220328.205775,26068,8.452056,350.205124
2,Trigram,-240018.211437,25068,9.574685,762.548513
3,Quadrigram,-259707.388617,24068,10.790568,1771.269372


## 3. Katz Backoff Smoothing

Katz backoff combines Good-Turing discounting with backoff:

    P_katz(w|h) = D(count) · count(h,w) / count(h)        if count(h,w) > 0
    P_katz(w|h) = alpha(h) · P_katz(w|h')                 otherwise

- Counts 1..4 are Good-Turing discounted (capped, cutoff 5); counts 5+ are unchanged.
- The backoff weight collects the discounted mass:

    alpha(h) = Σ_{w: 1 ≤ count(h,w) < k} (count(h,w) − count*(h,w)) / count(h)

- If the context itself was never seen, the model backs off directly to the lower order.
- alpha is precomputed **only for contexts that actually occur in the dev/test sets** (lazy precomputation),
  which avoids building an index over all ~18M contexts.

In [17]:
KATZ_THRESHOLD = 5

# Contexts that occur in the evaluation sets, per order.
KATZ_NEEDED_CONTEXTS = {2: set(), 3: set(), 4: set()}
for sentence in dev_sentences + test_sentences:
    for i in range(1, len(sentence)):
        if i >= 1:
            KATZ_NEEDED_CONTEXTS[2].add((sentence[i - 1],))
        if i >= 2:
            KATZ_NEEDED_CONTEXTS[3].add((sentence[i - 2], sentence[i - 1]))
        if i >= 3:
            KATZ_NEEDED_CONTEXTS[4].add((sentence[i - 3], sentence[i - 2], sentence[i - 1]))

print("Contexts needed:", {n: len(s) for n, s in KATZ_NEEDED_CONTEXTS.items()})

Contexts needed: {2: 8559, 3: 31738, 4: 41244}


In [18]:
# Sum of raw counts and GT-discounted counts of low-count (1..4) types, per needed context.
KATZ_LOW_SUMS = {2: {}, 3: {}, 4: {}}

for n in (2, 3, 4):
    needed = KATZ_NEEDED_CONTEXTS[n]
    sums = KATZ_LOW_SUMS[n]
    for ngram, count in COUNT_MAPS[n].items():
        if 1 <= count < KATZ_THRESHOLD:
            context = ngram[:-1]
            if context in needed:
                entry = sums.setdefault(context, [0.0, 0.0])
                entry[0] += count
                entry[1] += gt_adjusted_count(n, count)

print("Contexts with low-count types:", {n: len(s) for n, s in KATZ_LOW_SUMS.items()})

Contexts with low-count types: {2: 8072, 3: 25241, 4: 20395}


In [19]:
def katz_alpha(n, context):
    context_count = CONTEXT_COUNTS[n].get(context, 0)
    if context_count == 0:
        return 1.0
    raw, discounted = KATZ_LOW_SUMS[n].get(context, (0.0, 0.0))
    return max((raw - discounted) / context_count, 1e-12)


def katz_probability(n, ngram):
    if n == 1:
        return unigram_probability(ngram[0])

    context_count = CONTEXT_COUNTS[n].get(ngram[:-1], 0)
    if context_count == 0:
        return katz_probability(n - 1, ngram[1:])

    count = COUNT_MAPS[n].get(ngram, 0)
    if count > 0:
        adjusted = count if count >= KATZ_THRESHOLD else gt_adjusted_count(n, count)
        return max(adjusted, 1e-300) / context_count

    return katz_alpha(n, ngram[:-1]) * katz_probability(n - 1, ngram[1:])

In [20]:
def evaluate_katz(sentences, n):
    total = 0.0
    tokens = 0
    for sentence in sentences:
        for i in range(n - 1, len(sentence)):
            ngram = tuple(sentence[i - n + 1:i + 1])
            total += math.log2(max(katz_probability(n, ngram), 1e-300))
            tokens += 1
    ce = -total / tokens
    return total, tokens, ce, 2 ** ce


KATZ_MODELS = [
    ("Unigram", 1),
    ("Bigram", 2),
    ("Trigram", 3),
    ("Quadrigram", 4),
]


def results_table_katz(sentences):
    rows = []
    for name, n in KATZ_MODELS:
        log_p, tokens, ce, perplexity = evaluate_katz(sentences, n)
        rows.append({
            "Model": name,
            "Log Probability": log_p,
            "Tokens": tokens,
            "Cross Entropy": ce,
            "Perplexity": perplexity,
        })
    return pd.DataFrame(rows)

In [21]:
katz_dev_results = results_table_katz(dev_sentences)
katz_dev_results.to_csv("results/katz_development_results.csv", index=False)
katz_dev_results

,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Unigram,-260042.748092,24954,10.420884,1370.877887
1,Bigram,-193002.481058,23954,8.057213,266.356197
2,Trigram,-175759.752762,22954,7.657042,201.836386
3,Quadrigram,-172253.435114,21954,7.846107,230.098392


In [22]:
katz_test_results = results_table_katz(test_sentences)
katz_test_results.to_csv("results/katz_test_results.csv", index=False)
katz_test_results

,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Unigram,-286410.827649,27068,10.581160,1531.956466
1,Bigram,-211748.782289,26068,8.122939,278.771511
2,Trigram,-193121.419733,25068,7.703902,208.499798
3,Quadrigram,-189319.641011,24068,7.866031,233.298187


## 4. Stupid Backoff

Stupid Backoff (Brants & Franz, 2009) uses an **unnormalized** score S instead of a probability:

    S(w|h) = count(h,w) / count(h)        if count(h,w) > 0
    S(w|h) = 0.4 · S(w|h')                otherwise

with discount factor 0.4, bottoming out at the unigram estimate. Because the scores do not sum to 1, the
reported "perplexity" is 2^(average −log2 score) — useful for comparing models, but not a true
probability-domain perplexity. The unigram base uses the shared unseen floor so OOV tokens stay finite.

In [23]:
STUPID_ALPHA = 0.4


def stupid_score(n, ngram):
    if n == 1:
        return unigram_probability(ngram[0])

    context_count = CONTEXT_COUNTS[n].get(ngram[:-1], 0)
    if context_count > 0:
        count = COUNT_MAPS[n].get(ngram, 0)
        if count > 0:
            return count / context_count

    return STUPID_ALPHA * stupid_score(n - 1, ngram[1:])


def evaluate_stupid(sentences, n):
    total = 0.0
    tokens = 0
    for sentence in sentences:
        for i in range(n - 1, len(sentence)):
            ngram = tuple(sentence[i - n + 1:i + 1])
            total += math.log2(max(stupid_score(n, ngram), 1e-300))
            tokens += 1
    ce = -total / tokens
    return total, tokens, ce, 2 ** ce


STUPID_MODELS = [
    ("Unigram", 1),
    ("Bigram", 2),
    ("Trigram", 3),
    ("Quadrigram", 4),
]


def results_table_stupid(sentences):
    rows = []
    for name, n in STUPID_MODELS:
        log_p, tokens, ce, perplexity = evaluate_stupid(sentences, n)
        rows.append({
            "Model": name,
            "Log Probability": log_p,
            "Tokens": tokens,
            "Cross Entropy": ce,
            "Perplexity": perplexity,
        })
    return pd.DataFrame(rows)

In [24]:
stupid_dev_results = results_table_stupid(dev_sentences)
stupid_dev_results.to_csv("results/stupid_development_results.csv", index=False)
stupid_dev_results

,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Unigram,-260042.748092,24954,10.420884,1370.877887
1,Bigram,-187338.580472,23954,7.820764,226.091663
2,Trigram,-170588.067976,22954,7.431736,172.653527
3,Quadrigram,-179849.294197,21954,8.192097,292.460271


In [25]:
stupid_test_results = results_table_stupid(test_sentences)
stupid_test_results.to_csv("results/stupid_test_results.csv", index=False)
stupid_test_results

,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Unigram,-286410.827649,27068,10.581160,1531.956466
1,Bigram,-205812.775750,26068,7.895227,238.067510
2,Trigram,-187984.115663,25068,7.498967,180.889823
3,Quadrigram,-198123.685116,24068,8.231830,300.626844


## 5. Kneser-Ney Smoothing

Kneser-Ney discounts seen counts by a fixed D (estimated from count-of-counts) and backs off using
**continuation probabilities** — how many distinct contexts a word follows — instead of raw frequencies:

    P_KN(w|h) = max(count(h,w) − D, 0) / count(h) + λ(h) · P_KN(w|h')
    λ(h) = D · |{w : count(h,w) > 0}| / count(h)
    P_cont(w) = |{h : count(h,w) > 0}| / |{(h,w) : count(h,w) > 0}|        (base case, n = 1)

The discount is `D = N1 / (N1 + 2·N2)` computed on bigram counts. Distinct-continuation types per context
are precomputed lazily, only for contexts occurring in the evaluation sets.

In [26]:
KN_D_N1 = COUNT_OF_COUNTS[2].get(1, 0)
KN_D_N2 = COUNT_OF_COUNTS[2].get(2, 0)
KN_D = KN_D_N1 / (KN_D_N1 + 2 * KN_D_N2) if (KN_D_N1 + 2 * KN_D_N2) > 0 else 0.75

# Base continuation counts: number of distinct left contexts each word follows.
KN_CONTINUATION = Counter()
for (w1, w2) in bigram_counts:
    KN_CONTINUATION[w2] += 1

KN_TOTAL_TYPES = len(bigram_counts)

# Contexts that occur in the evaluation sets, per order.
KN_NEEDED_CONTEXTS = {2: set(), 3: set(), 4: set()}
for sentence in dev_sentences + test_sentences:
    for i in range(1, len(sentence)):
        if i >= 1:
            KN_NEEDED_CONTEXTS[2].add((sentence[i - 1],))
        if i >= 2:
            KN_NEEDED_CONTEXTS[3].add((sentence[i - 2], sentence[i - 1]))
        if i >= 3:
            KN_NEEDED_CONTEXTS[4].add((sentence[i - 3], sentence[i - 2], sentence[i - 1]))

print(f"Kneser-Ney discount D = {KN_D:.4f}")

Kneser-Ney discount D = 0.7396


In [27]:
# Distinct continuation types per needed context: |{w : count(h,w) > 0}|.
KN_TYPES_PER_CONTEXT = {2: Counter(), 3: Counter(), 4: Counter()}

for n in (2, 3, 4):
    needed = KN_NEEDED_CONTEXTS[n]
    types = KN_TYPES_PER_CONTEXT[n]
    for ngram in COUNT_MAPS[n]:
        context = ngram[:-1]
        if context in needed:
            types[context] += 1

print("Contexts indexed:", {n: len(t) for n, t in KN_TYPES_PER_CONTEXT.items()})

Contexts indexed: {2: 8115, 3: 25537, 4: 21059}


In [28]:
def kn_probability(n, ngram):
    if n == 1:
        continuation = KN_CONTINUATION.get(ngram[0], 0)
        if continuation > 0:
            return continuation / KN_TOTAL_TYPES
        return UNSEEN_PROBABILITY[1]

    context_count = CONTEXT_COUNTS[n].get(ngram[:-1], 0)
    count = COUNT_MAPS[n].get(ngram, 0)

    if context_count > 0:
        first_term = max(count - KN_D, 0) / context_count
        types = KN_TYPES_PER_CONTEXT[n].get(ngram[:-1], 0)
        lambda_h = KN_D * types / context_count
    else:
        first_term = 0.0
        lambda_h = 1.0

    return first_term + lambda_h * kn_probability(n - 1, ngram[1:])

In [29]:
def evaluate_kn(sentences, n):
    total = 0.0
    tokens = 0
    for sentence in sentences:
        for i in range(n - 1, len(sentence)):
            ngram = tuple(sentence[i - n + 1:i + 1])
            total += math.log2(max(kn_probability(n, ngram), 1e-300))
            tokens += 1
    ce = -total / tokens
    return total, tokens, ce, 2 ** ce


KN_MODELS = [
    ("Unigram", 1),
    ("Bigram", 2),
    ("Trigram", 3),
    ("Quadrigram", 4),
]


def results_table_kn(sentences):
    rows = []
    for name, n in KN_MODELS:
        log_p, tokens, ce, perplexity = evaluate_kn(sentences, n)
        rows.append({
            "Model": name,
            "Log Probability": log_p,
            "Tokens": tokens,
            "Cross Entropy": ce,
            "Perplexity": perplexity,
        })
    return pd.DataFrame(rows)

In [30]:
kn_dev_results = results_table_kn(dev_sentences)
kn_dev_results.to_csv("results/kneser_ney_development_results.csv", index=False)
kn_dev_results

,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Unigram,-307984.663090,24954,12.342096,5192.076699
1,Bigram,-189285.864500,23954,7.902057,239.197190
2,Trigram,-168557.232727,22954,7.343262,162.383570
3,Quadrigram,-162535.125258,21954,7.403440,169.300234


In [31]:
kn_test_results = results_table_kn(test_sentences)
kn_test_results.to_csv("results/kneser_ney_test_results.csv", index=False)
kn_test_results

,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Unigram,-334442.789185,27068,12.355652,5241.093137
1,Bigram,-207390.645422,26068,7.955756,248.268243
2,Trigram,-185068.930268,25068,7.382676,166.881050
3,Quadrigram,-178447.475595,24068,7.414304,170.579952


## Final Comparison

All results across the 5 smoothing techniques on both evaluation sets.

In [32]:
result_files = {
    "Interpolated": ("results/interpolated_development_results.csv", "results/interpolated_test_results.csv"),
    "Good Turing": ("results/good_turing_development_results.csv", "results/good_turing_test_results.csv"),
    "Katz Backoff": ("results/katz_development_results.csv", "results/katz_test_results.csv"),
    "Stupid Backoff": ("results/stupid_development_results.csv", "results/stupid_test_results.csv"),
    "Kneser-Ney": ("results/kneser_ney_development_results.csv", "results/kneser_ney_test_results.csv"),
}

frames = []
for technique, (dev_file, test_file) in result_files.items():
    dev_frame = pd.read_csv(dev_file)
    dev_frame.insert(0, "Technique", technique)
    dev_frame.insert(0, "Dataset", "Development")
    frames.append(dev_frame)

    test_frame = pd.read_csv(test_file)
    test_frame.insert(0, "Technique", technique)
    test_frame.insert(0, "Dataset", "Test")
    frames.append(test_frame)

all_results = pd.concat(frames, ignore_index=True)
all_results.to_csv("results/all_smoothing_results.csv", index=False)

all_results

,Dataset,Technique,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Development,Interpolated,Bigram,-194638.425378,23954,8.125508,279.268352
1,Development,Interpolated,Trigram,-175142.822208,22954,7.630166,198.111064
2,Development,Interpolated,Quadrigram,-167660.783670,21954,7.636913,199.039755
3,Test,Interpolated,Bigram,-213882.097629,26068,8.204776,295.041867
4,Test,Interpolated,Trigram,-193158.577227,25068,7.705384,208.714127
5,Test,Interpolated,Quadrigram,-185359.706540,24068,7.701500,208.152949
6,Development,Good Turing,Unigram,-260042.748092,24954,10.420884,1370.877887
7,Development,Good Turing,Bigram,-200405.346086,23954,8.366258,329.985331
8,Development,Good Turing,Trigram,-217567.384557,22954,9.478408,713.321337
9,Development,Good Turing,Quadrigram,-236550.116336,21954,10.774807,1752.024407


In [33]:
dev_only = all_results[all_results["Dataset"] == "Development"]
test_only = all_results[all_results["Dataset"] == "Test"]

print("Best model on Development set:")
print(dev_only.loc[dev_only["Perplexity"].idxmin(), ["Technique", "Model", "Perplexity"]].to_string())
print()
print("Best model on Test set:")
print(test_only.loc[test_only["Perplexity"].idxmin(), ["Technique", "Model", "Perplexity"]].to_string())
print()

pivot = all_results.pivot_table(index=["Technique", "Model"], columns="Dataset", values="Perplexity")
pivot = pivot[["Development", "Test"]].round(2)
pivot

Best model on Development set:
Technique     Kneser-Ney
Model            Trigram
Perplexity     162.38357

Best model on Test set:
Technique     Kneser-Ney
Model            Trigram
Perplexity     166.88105



Dataset                    Development     Test
Technique      Model                           
Good Turing    Bigram           329.99   350.21
               Quadrigram      1752.02  1771.27
               Trigram          713.32   762.55
               Unigram         1370.88  1531.96
Interpolated   Bigram           279.27   295.04
               Quadrigram       199.04   208.15
               Trigram          198.11   208.71
Katz Backoff   Bigram           266.36   278.77
               Quadrigram       230.10   233.30
               Trigram          201.84   208.50
               Unigram         1370.88  1531.96
Kneser-Ney     Bigram           239.20   248.27
               Quadrigram       169.30   170.58
               Trigram          162.38   166.88
               Unigram         5192.08  5241.09
Stupid Backoff Bigram           226.09   238.07
               Quadrigram       292.46   300.63
               Trigram          172.65   180.89
               Unigram         1370.88  1531.96

In [34]:
print("Assignment 6 completed")
print()
print("Smoothing techniques implemented:")
print("1. Interpolated Smoothing (Bigram, Trigram, Quadrigram) - weights tuned on dev set")
print("2. Good Turing Smoothing - count-of-counts rescaling, cutoff 5")
print("3. Katz Backoff Smoothing - GT discounting with alpha backoff")
print("4. Stupid Backoff - unnormalized scores, discount 0.4")
print("5. Kneser-Ney Smoothing - absolute discounting with continuation probabilities")
print()
print(f"Vocabulary size: {V:,}")
print(f"Development sentences: {len(dev_df):,}")
print(f"Test sentences: {len(test_df):,}")
print()
print("Results saved to results/all_smoothing_results.csv")

Assignment 6 completed

Smoothing techniques implemented:
1. Interpolated Smoothing (Bigram, Trigram, Quadrigram) - weights tuned on dev set
2. Good Turing Smoothing - count-of-counts rescaling, cutoff 5
3. Katz Backoff Smoothing - GT discounting with alpha backoff
4. Stupid Backoff - unnormalized scores, discount 0.4
5. Kneser-Ney Smoothing - absolute discounting with continuation probabilities

Vocabulary size: 307,861
Development sentences: 1,000
Test sentences: 1,000

Results saved to results/all_smoothing_results.csv
